In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# HorusEye Q1 level 2 - Posture assessment + calibration
# Qwen3-VL, full + cropped, clean/fog/smoke
# accuracy + ECE/AUROC, global AND per posture class (LYING priority)
# Environment: Kaggle Notebook, GPU T4.
# Kaggle settings: GPU T4 ON, Internet ON, add your posture dataset as input.
#
# Two separate image folders (routed by 'source' field):
#   - rq3_original_50   -> DIR_50   (short names: 000002.jpg)
#   - lying_candidates  -> DIR_170  (long names: COCO_train2014_...jpg)
# Each folder has clean/ fog/ smoke/ subfolders.

In [1]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)
pip("-U", "transformers", "accelerate")
pip("qwen-vl-utils")

import os, re, json, gc
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- CONFIG (adjust to your Kaggle dataset paths) ----------
UNIFIED_DATASET = "/kaggle/input/datasets/armelyara/posture-data/posture_dataset/posture_dataset_unified.json"
DIR_50  = "/kaggle/input/datasets/armelyara/refcoco-d/refcoco_degraded_benchmark/images"   # 50 originals, short names
DIR_170 = "/kaggle/input/datasets/armelyara/posture-data/posture_dataset/posture_degraded"   # 170 new, long names
OUTPUT_DIR = "/kaggle/working/q1_posture_results"

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"   # falls back note below if unavailable
CONDITIONS = ["clean", "fog", "smoke"]
VIEWS = ["full", "cropped"]
LABELS = ["STANDING", "SITTING", "LYING"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# route each image to its folder based on 'source'
def image_dir_for(source):
    return DIR_50 if source == "rq3_original_50" else DIR_170

# ---------- PROMPT / PARSING ----------
PROMPT = (
    "You are analyzing an image of a person in an emergency scene. "
    "Classify the person's posture as EXACTLY one of: [standing, sitting, lying]. "
    "A lying person (on the ground) is likely a high-priority victim. "
    "Give your confidence between 0.0 and 1.0. "
    "Respond ONLY with valid JSON, no markdown:\n"
    '{"posture": "<one of the 3>", "confidence": <float 0-1>}'
)

def parse_response(text):
    if not text:
        return None
    text = text.replace("```json", "").replace("```", "").strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        o = json.loads(m.group(0))
        p = str(o.get("posture", "")).strip().upper()
        if p not in LABELS:
            for lab in LABELS:
                if lab[:4].lower() in p.lower():
                    p = lab
                    break
        conf = float(o.get("confidence", 0.0))
        conf = min(max(conf, 0.0), 1.0)
        return {"posture": p if p in LABELS else "UNKNOWN", "confidence": conf}
    except Exception:
        return None

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.7 MB/s eta 0:00:00


In [2]:
# ---------- MODEL ----------
print("Loading Qwen3-VL...")
try:
    from transformers import Qwen3VLForConditionalGeneration as VLModel
except Exception:
    # fallback name if the class differs in the installed transformers version
    from transformers import AutoModelForImageTextToText as VLModel
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

model = VLModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME}")

def qwen_predict(pil_image, retries=1):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pil_image},
        {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    try:
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        out = processor.batch_decode(trimmed, skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)[0]
        del inputs, gen, trimmed
        return out
    except torch.cuda.OutOfMemoryError:
        del inputs
        gc.collect(); torch.cuda.empty_cache()
        return qwen_predict(pil_image, retries - 1) if retries > 0 else "ERROR: OOM"
    finally:
        gc.collect(); torch.cuda.empty_cache()

# ---------- IMAGE LOADING (full + cropped) ----------
def load_image(sample, condition, view):
    """Return a PIL image for the given sample/condition/view, or None."""
    root = image_dir_for(sample["source"])
    path = os.path.join(root, condition, sample["file_name"])
    if not os.path.exists(path):
        return None
    img = Image.open(path).convert("RGB")
    if view == "cropped":
        x, y, w, h = sample["bbox"]
        x, y, w, h = int(x), int(y), int(w), int(h)
        # clamp to image bounds
        x2, y2 = min(x + w, img.width), min(y + h, img.height)
        x, y = max(0, x), max(0, y)
        if x2 > x and y2 > y:
            img = img.crop((x, y, x2, y2))
    return img

# ---------- METRICS ----------
def compute_f1(records):
    idx = {l: i for i, l in enumerate(LABELS)}
    cm = np.zeros((3, 3), dtype=int)
    correct = valid = 0
    for r in records:
        t_i = idx[r["true"]]
        p = r["pred"]
        if p in idx:
            cm[t_i][idx[p]] += 1
            valid += 1
            if idx[p] == t_i:
                correct += 1
    acc = correct / valid if valid else 0.0
    f1s = {}
    for i, l in enumerate(LABELS):
        tp = cm[i][i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        pr = tp / (tp + fp) if (tp + fp) else 0.0
        rc = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
        f1s[l] = {"precision": pr, "recall": rc, "f1": f1, "support": int(cm[i].sum())}
    present = [l for l in LABELS if f1s[l]["support"] > 0]
    f1_macro = float(np.mean([f1s[l]["f1"] for l in present])) if present else 0.0
    return acc, f1_macro, f1s, cm

def compute_ece(confidences, correctness, n_bins=10):
    confidences = np.array(confidences); correctness = np.array(correctness)
    if len(confidences) == 0:
        return 0.0
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0; N = len(confidences)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / N) * abs(correctness[mask].mean() - confidences[mask].mean())
    return float(ece)

def compute_auroc(confidences, correctness):
    confidences = np.array(confidences); correctness = np.array(correctness)
    pos = confidences[correctness == 1]; neg = confidences[correctness == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    count = 0.0
    for p in pos:
        count += np.sum(p > neg) + 0.5 * np.sum(p == neg)
    return float(count / (len(pos) * len(neg)))

def signed_gap(confidences, correctness):
    """Mean confidence - mean accuracy. Positive = over-confident."""
    if not confidences:
        return 0.0
    return float(np.mean(confidences) - np.mean(correctness))



Loading Qwen3-VL...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
Loaded Qwen/Qwen3-VL-8B-Instruct


In [3]:
# ---------- MAIN ----------
data = json.load(open(UNIFIED_DATASET))
samples = data["samples"]
print(f"Samples: {len(samples)} | views: {VIEWS} | conditions: {CONDITIONS}")

# records[view][condition] = list of {true, pred, confidence, correct}
records = {v: {c: [] for c in CONDITIONS} for v in VIEWS}
missing = 0

for s in tqdm(samples, desc="Q1 level2 posture"):
    true = s["posture"]
    for view in VIEWS:
        for cond in CONDITIONS:
            img = load_image(s, cond, view)
            if img is None:
                missing += 1
                continue
            raw = qwen_predict(img)
            parsed = parse_response(raw)
            if parsed is None:
                pred, conf = "UNKNOWN", 0.0
            else:
                pred, conf = parsed["posture"], parsed["confidence"]
            records[view][cond].append({
                "file_name": s["file_name"], "true": true, "pred": pred,
                "confidence": conf, "correct": int(pred == true),
            })

print(f"\nMissing image loads: {missing}")

# ---------- ANALYSIS ----------
summary = {}
for view in VIEWS:
    summary[view] = {}
    for cond in CONDITIONS:
        recs = records[view][cond]
        if not recs:
            continue
        acc, f1m, f1s, cm = compute_f1(recs)
        confs = [r["confidence"] for r in recs]
        corr = [r["correct"] for r in recs]
        ece = compute_ece(confs, corr)
        auroc = compute_auroc(confs, corr)
        gap = signed_gap(confs, corr)

        # per-class calibration (LYING priority)
        per_class = {}
        for lab in LABELS:
            lab_recs = [r for r in recs if r["true"] == lab]
            if lab_recs:
                lc = [r["confidence"] for r in lab_recs]
                lk = [r["correct"] for r in lab_recs]
                per_class[lab] = {
                    "n": len(lab_recs),
                    "accuracy": float(np.mean(lk)),
                    "mean_confidence": float(np.mean(lc)),
                    "signed_gap": signed_gap(lc, lk),
                    "ece": compute_ece(lc, lk),
                }

        summary[view][cond] = {
            "n": len(recs), "accuracy": acc, "f1_macro": f1m,
            "ece": ece, "auroc": auroc, "signed_gap": gap,
            "per_class": {**f1s},
            "per_class_calibration": per_class,
            "confusion_matrix": cm.tolist(),
        }

# save records + summary
with open(f"{OUTPUT_DIR}/posture_all_records.json", "w") as f:
    json.dump(records, f, indent=2)
with open(f"{OUTPUT_DIR}/posture_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ---------- FIGURES: reliability full vs cropped ----------
def reliability_points(recs, n_bins=10):
    confs = np.array([r["confidence"] for r in recs])
    corr = np.array([r["correct"] for r in recs])
    bins = np.linspace(0, 1, n_bins + 1); xs, ys = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confs > lo) & (confs <= hi) if i > 0 else (confs >= lo) & (confs <= hi)
        if mask.sum() == 0:
            continue
        xs.append(confs[mask].mean()); ys.append(corr[mask].mean())
    return xs, ys

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
colors = {"full": "#2196F3", "cropped": "#E64A19"}
for view in VIEWS:
    allrecs = [r for c in CONDITIONS for r in records[view][c]]
    if allrecs:
        xs, ys = reliability_points(allrecs)
        ax.plot(xs, ys, "o-", color=colors[view], label=f"{view}")
ax.set_xlabel("Stated confidence"); ax.set_ylabel("Empirical accuracy")
ax.set_title("Q1 level 2 - Posture reliability (full vs cropped)")
ax.legend(); ax.grid(alpha=.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/posture_reliability_full_vs_cropped.png", dpi=150, bbox_inches="tight")
plt.close()

# ---------- PRINT RECAP ----------
print("\n" + "=" * 66)
print("Q1 LEVEL 2 - POSTURE CALIBRATION RECAP")
print("=" * 66)
for view in VIEWS:
    print(f"\n[{view.upper()}]")
    print(f"  {'cond':<8}{'n':<5}{'acc':<7}{'F1':<7}{'ECE':<7}{'AUROC':<7}{'gap':<8}")
    for cond in CONDITIONS:
        if cond in summary[view]:
            s = summary[view][cond]
            print(f"  {cond:<8}{s['n']:<5}{s['accuracy']:<7.3f}{s['f1_macro']:<7.3f}"
                  f"{s['ece']:<7.3f}{s['auroc']:<7.3f}{s['signed_gap']:<+8.3f}")
    # LYING focus
    print(f"  -- LYING (critical triage class) --")
    for cond in CONDITIONS:
        if cond in summary[view]:
            ly = summary[view][cond]["per_class_calibration"].get("LYING")
            if ly:
                print(f"    {cond:<8} n={ly['n']:<3} acc={ly['accuracy']:.3f} "
                      f"conf={ly['mean_confidence']:.3f} gap={ly['signed_gap']:+.3f}")

import shutil
shutil.make_archive("/kaggle/working/q1_posture_results", "zip", "/kaggle/working/q1_posture_results")
print("\nDone. Download /kaggle/working/q1_posture_results.zip")
print("\nKey questions this answers:")
print("  1. Does posture calibration degrade under veils (like level 1)?")
print("  2. Is the model over-confident on LYING people (deadly triage error)?")
print("  3. Does cropping help or hurt calibration (vs accuracy)?")

Samples: 220 | views: ['full', 'cropped'] | conditions: ['clean', 'fog', 'smoke']


Q1 level2 posture: 100%|██████████| 220/220 [36:55<00:00, 10.07s/it]



Missing image loads: 200

Q1 LEVEL 2 - POSTURE CALIBRATION RECAP

[FULL]
  cond    n    acc    F1     ECE    AUROC  gap     
  clean   220  0.809  0.812  0.174  0.671  +0.174  
  fog     170  0.776  0.750  0.204  0.610  +0.204  
  smoke   170  0.747  0.714  0.230  0.584  +0.230  
  -- LYING (critical triage class) --
    clean    n=56  acc=0.929 conf=0.988 gap=+0.059
    fog      n=54  acc=0.907 conf=0.986 gap=+0.078
    smoke    n=54  acc=0.870 conf=0.982 gap=+0.112

[CROPPED]
  cond    n    acc    F1     ECE    AUROC  gap     
  clean   220  0.777  0.772  0.195  0.699  +0.195  
  fog     170  0.747  0.690  0.219  0.687  +0.219  
  smoke   170  0.682  0.613  0.281  0.703  +0.281  
  -- LYING (critical triage class) --
    clean    n=56  acc=0.964 conf=0.983 gap=+0.019
    fog      n=54  acc=0.963 conf=0.976 gap=+0.013
    smoke    n=54  acc=0.889 conf=0.974 gap=+0.085

Done. Download /kaggle/working/q1_posture_results.zip

Key questions this answers:
  1. Does posture calibration deg